# Guided rewrites: the student fixes the words a persona flagged

`training/persona_judge.py drafts` had Claude read the student's best briefing for each training packet as one of the harness personas and flag the words that persona would not follow (`not_understood`) and the explanations it did not need (`needlessly_explained`). Claude wrote no briefing text.

Here the student rewrites each flagged draft, told exactly which words to fix (`training/prompts/briefing_guided_revise.md`). The student can do that; what it cannot do is spot the words itself. Back in the repo, `build_pairs_guided.py` keeps each rewrite that fixed the flagged words without breaking a rule or dropping facts, as (rewrite, draft) DPO pairs for round 3.

**Runtime → Change runtime type → L4 GPU, then Runtime → Run all.** About 30 minutes.


In [ ]:
import subprocess, sys, importlib

USE_UNSLOTH = True  # the fallback cell sets this to False

def pip(*args):
    cmd = [sys.executable, "-m", "pip", "install", "--quiet", *args]
    print("$", " ".join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout[-3000:], result.stderr[-3000:])
        raise RuntimeError("pip failed -- run the fallback cell below instead")

# unsloth pins its own tested versions of trl / peft / transformers / accelerate /
# bitsandbytes / datasets, so we pin unsloth to a release series and let it choose.
pip("--upgrade", "unsloth==2025.9.*", "unsloth_zoo==2025.9.*")
pip("matplotlib")

print()
print("installed versions:")
for name in ["unsloth", "trl", "peft", "transformers", "datasets", "accelerate", "bitsandbytes", "torch"]:
    try:
        mod = importlib.import_module(name)
        print(f"  {name:<14} {getattr(mod, '__version__', '?')}")
    except Exception as exc:  # noqa: BLE001
        print(f"  {name:<14} NOT IMPORTABLE ({exc})")

## 2. Drive, the round-2 adapter and the flags


In [ ]:
import json, os, urllib.request
from google.colab import drive
drive.mount("/content/drive", timeout_ms=1800000)  # waits up to 30 minutes for the Drive permission click
WORK_DIR = "/content/drive/MyDrive/news-briefer"
ADAPTER_NAME = "news-briefer-r2-lora"
FLAGS_NAME = "persona_flags.jsonl"
OUT_NAME = "guided_revisions.jsonl"
ADAPTER_DIR = os.path.join(WORK_DIR, ADAPTER_NAME)
assert os.path.isdir(ADAPTER_DIR), f"adapter not found at {ADAPTER_DIR}"
REL = "https://github.com/sanjivp2703/adaptive-news-briefing-agent/releases/download/training-data-v7/"
urllib.request.urlretrieve(REL + FLAGS_NAME, FLAGS_NAME)
flags = [json.loads(l) for l in open(FLAGS_NAME) if l.strip()]
todo = [f for f in flags if f["not_understood"] or f["needlessly_explained"]]
raw = urllib.request.urlopen("https://raw.githubusercontent.com/sanjivp2703/adaptive-news-briefing-agent/main/training/prompts/briefing_guided_revise.md").read().decode()
PROMPT = raw.split("---", 2)[2].strip() if raw.startswith("---") else raw
print(len(flags), "flagged drafts,", len(todo), "with something to fix")


## 3. Rewrite


In [ ]:
import torch
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(model_name=ADAPTER_DIR, max_seq_length=8192, dtype=None, load_in_4bit=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
FastLanguageModel.for_inference(model)

def as_json(text):
    a, b = text.find("{"), text.rfind("}")
    if a < 0 or b <= a:
        return None
    try:
        obj = json.loads(text[a:b + 1])
    except ValueError:
        return None
    return obj if isinstance(obj, dict) and str(obj.get("briefing") or "").strip() else None

out_path = os.path.join(WORK_DIR, OUT_NAME)
done = set()
if os.path.exists(out_path):  # resumable after a disconnect
    done = {json.loads(l)["id"] for l in open(out_path) if l.strip()}
with open(out_path, "a", encoding="utf-8") as out:
    for i, f in enumerate(todo, 1):
        if f["id"] in done:
            continue
        draft = as_json(f["draft"])
        if draft is None:
            continue
        user = json.dumps({"draft": draft, "lost": f["not_understood"], "already_knew": f["needlessly_explained"]}, ensure_ascii=False)
        ids = tokenizer.apply_chat_template([{"role": "system", "content": PROMPT}, {"role": "user", "content": user}],
                                           add_generation_prompt=True, return_tensors="pt").to(model.device)
        revisions = []
        with torch.no_grad():
            gen = model.generate(input_ids=ids.repeat(3, 1), max_new_tokens=900, temperature=0.5, do_sample=True, top_p=0.95,
                                 pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
        for g in gen:
            fixed = as_json(tokenizer.decode(g[ids.shape[1]:], skip_special_tokens=True))
            if fixed is not None:
                fixed.setdefault("reasoning", draft.get("reasoning", ""))
                revisions.append(json.dumps(fixed, ensure_ascii=False))
        out.write(json.dumps({"id": f["id"], "draft": f["draft"], "revisions": revisions,
                              "not_understood": f["not_understood"], "needlessly_explained": f["needlessly_explained"]}, ensure_ascii=False) + "\n")
        out.flush()
        print(f"  {i}/{len(todo)}  {len(revisions)} revisions")
print("wrote", out_path)


## 4. Download


In [ ]:
from google.colab import files
files.download(os.path.join(WORK_DIR, OUT_NAME))
